# Use Case Onboarding

Sample notebook demonstrating how to utilize the deployed functions to:
- Create a new index
- List all files within a target Azure Storage container
- Trigger ingestion and indexing of that collection of files into the target storage index
- Retrieve a list of all chunks which have been added into the target index

Note: Before running this notebook, you will need to have deployed the Azure Durable Functions project to an Azure Function App environment as well as all associated resources (Azure AI Search, Azure Storage, Azure OpenAI, Azure Document Intelligence).

### Environment Variable Configuration

Create a `.env` file in your working directory with the following key-value pairs. We will load these into code using the `python-dotenv` library. 

| Variable       | Description                                                                                 |  
|----------------|---------------------------------------------------------------------------------------------|  
| FUNCTION_URI   | The primary URI endpoint for accessing your deployed Azure Function App.                    |  
| FUNCTION_KEY   | The default host key used for authenticating and securing access to your Azure Function App. | 

In [1]:
import os
from dotenv import load_dotenv
import requests
import json
from IPython.display import clear_output
import time
import pandas as pd

load_dotenv(override=True)

function_uri = os.getenv("FUNCTION_URI")
function_key = os.getenv("FUNCTION_KEY")

### Define Execution Variables

Below, create a stem name for a new Azure AI Search Index and fields that will be used in index creation. Further, define Azure Storage containers which contain source documents and which will contain extracted chunks, respectively. 

In [ ]:
# Index Creation Settings
index_stem_name = 'test-index'
fields = {
    "content": "string", "pagenumber": "int", "sourcefile": "string", 
    "sourcefilepath": "string","sourcepage": "string", "category": "string"
}
embedding_dimensions = 3072 # Update according to the embedding model used (3072 for text-embedding-large-003; 1536 for text-embedding-ada-002)

# Data Source Settings
source_container = 'a-test-source'
extract_container = 'a-test-extract'

# Ingestion Settings
automatically_delete = False
analyze_images = True
chunking_strategy = 'semantic'
max_chunk_size = 1200
chunk_overlap = 200
embedding_model = 'text-embedding-large-003'  # Update according to the name of your embedding model deployment 
cosmos_logging = False

### Step 1 - Create New Index

Trigger the `create_new_index` function and store the created index name

In [ ]:
create_index_uri = f"{function_uri}/api/create_new_index?code={function_key}"
create_index_payload = {
    "index_stem_name": index_stem_name,
    "fields": fields,
    "dimensions": embedding_dimensions
}

response = requests.post(create_index_uri, json=create_index_payload)
index_name = response.text
print(index_name)

### Step 2 - List Files in Source Container

In [ ]:
list_files_uri = f"{function_uri}/api/list_files_in_container?code={function_key}"

response = requests.post(list_files_uri, json={"container": source_container})
files = response.json()
files

### Step 3 - Trigger Ingestion for all Files and Await Completion

In [ ]:
def start_processing(blob_name, index_name):
    body_template = {
        'source_container': source_container,
        'extract_container': extract_container,
        'prefix_path':'',
        'index_name': index_name,
        'automatically_delete': automatically_delete,
        'analyze_images': analyze_images,
        'chunking_strategy': chunking_strategy,
        'max_chunk_size': max_chunk_size,
        'chunk_overlap': chunk_overlap,
        'cosmos_logging': cosmos_logging
    }

    body = body_template.copy()
    body['prefix_path'] = blob_name
    function_uri = f'{os.environ["FUNCTION_URI"]}/api/orchestrators/pdf_orchestrator?code={os.environ["FUNCTION_KEY"]}'
    response = requests.post(function_uri, json=(body))
    return response.json()['statusQueryGetUri']

def get_status(status_uri):
    response = requests.get(status_uri)
    status = response.json()['runtimeStatus']
    error = ''
    if status == 'Failed':
        error = response.json()['output']
    return status, error


# Submit all files for ingestion
tracking_dict = {}

for blob in files:
    blob_name = blob
    tracking_dict[blob_name] = {}
    status_uri = start_processing(blob_name, index_name)
    tracking_dict[blob_name]['status_uri'] = status_uri
    tracking_dict[blob_name]['submitted'] = True
    tracking_dict[blob_name]['completed'] = False
    status, error = get_status(status_uri)
    tracking_dict[blob_name]['error'] = error
    tracking_dict[blob_name]['status'] = status

print(f'Submitted {str(len(tracking_dict))} blobs for processing')

while True:
    all_complete = True
    total_complete = 0
    for k,v in tracking_dict.items():
        status, error = get_status(v['status_uri'])
        v['status'] = status
        v['error'] = error
        if v['status'] == 'Completed':
            v['completed'] = True
            total_complete += 1
        elif v['status'] == 'Failed': 
            # Logic to proceed forward WITHOUT retrying failed blobs
            status, error = get_status(status_uri)
            v['error'] = error
            v['status'] = status
            v['completed'] = True
            total_complete +=1

            # Logic to retry failed blobs
            # status_uri = start_processing(k)
            # v['status_uri'] = status_uri
            # v['submitted'] = True
            # v['completed'] = False
        else:
            all_complete = False
    clear_output(wait=True)
    print(f'Completed: {total_complete}/{len(tracking_dict)}')
    display(pd.DataFrame(tracking_dict).T)
    if all_complete:
        break
    time.sleep(10)

clear_output(wait=True)
print('All Blobs Processed')
succeeded = len([k for k,v in tracking_dict.items() if v['status'] == 'Completed'])
failed = len([k for k,v in tracking_dict.items() if v['status'] != 'Completed'])

print(f'Succeeded: {succeeded}')
succeeded_dict = {k: v for k, v in tracking_dict.items() if v['status'] == 'Completed'}
display(pd.DataFrame(succeeded_dict).T)
print()
print(f'Failed: {failed}')
failed_dict = {k: v for k, v in tracking_dict.items() if v['status'] != 'Completed'}
display(pd.DataFrame(failed_dict).T)


### Step 3.5 (Optional) - Attempt to Resubmit Failed Files

Uncommend the logic below and execute

In [6]:
# for k,v in failed_dict.items():
#     status_uri = start_processing(k)
#     v['status_uri'] = status_uri
#     v['submitted'] = True
#     v['completed'] = False

# while True:
#     all_complete = True
#     total_complete = 0
#     for k,v in failed_dict.items():
#         status, error = get_status(v['status_uri'])
#         v['status'] = status
#         v['error'] = error
#         if v['status'] == 'Completed':
#             v['completed'] = True
#             total_complete += 1
#         elif v['status'] == 'Failed': # Resubmit
#             status_uri = start_processing(k)
#             v['status_uri'] = status_uri
#             v['submitted'] = True
#             v['completed'] = False
#             status, error = get_status(status_uri)
#             v['error'] = error
#             v['status'] = status
#             v['completed'] = True
#             total_complete +=1
#         else:
#             all_complete = False
#     clear_output(wait=True)
#     print(f'Completed: {total_complete}/{len(failed_dict)}')
#     display(pd.DataFrame(failed_dict).T)
#     if all_complete:
#         break
#     time.sleep(10)


### Step 4 - Run 'Sync Index' to Retrieve a List of all Indexed Content

In [ ]:
sync_index_uri = f"{function_uri}/api/orchestrators/sync_index_orchestrator?code={function_key}"
response = requests.post(sync_index_uri, json={"index_name": index_name, "extract_container": extract_container})
status_uri = response.json()['statusQueryGetUri']

while True:
    status, error = get_status(status_uri)
    if status == 'Completed':
        clear_output(wait=True)
        break
    clear_output(wait=True)
    print(f'Status: {status}')
    print(f'Error: {error}')
    time.sleep(10)

output = requests.get(status_uri)
index_content = output.json()['output']['index_content']
print(f'Indexed Content: {index_name}')
pd.DataFrame(index_content)
# Optional: save as CSV
# pd.DataFrame(index_content).to_csv(f'{index_name}.csv', index=False)

# Prints full list of indexed content
# print(json.dumps(output.json()['output']['index_content'], indent=2))

---

## File Upload Examples

The cells below demonstrate how to upload local files to Azure Blob Storage for ingestion, then trigger the full pipeline with test identity values.

### Prerequisites
Add the following to your `.env` file:

| Variable               | Description                                                  |
|------------------------|--------------------------------------------------------------|
| `STORAGE_CONN_STR`     | Azure Storage Account connection string                      |
| `SOURCE_CONTAINER`     | Blob container name for source documents (e.g. `pdf-content`)|

In [ ]:
from azure.storage.blob import BlobServiceClient, BlobClient
import os
import glob

# Load storage connection string
storage_conn_str = os.getenv("STORAGE_CONN_STR")
upload_container = os.getenv("SOURCE_CONTAINER", "pdf-content")

blob_service_client = BlobServiceClient.from_connection_string(storage_conn_str)
container_client = blob_service_client.get_container_client(upload_container)

# Create container if it doesn't exist
try:
    container_client.create_container()
    print(f"Created container: {upload_container}")
except Exception:
    print(f"Container '{upload_container}' already exists")

### Upload a Single File

Upload one local file to the source blob container.

In [ ]:
def upload_file(local_path, blob_prefix="", container=None):
    """Upload a single local file to Azure Blob Storage.
    
    Args:
        local_path: Path to the local file.
        blob_prefix: Optional folder prefix in the blob container.
        container: Container client (defaults to upload_container).
    
    Returns:
        The blob name of the uploaded file.
    """
    if container is None:
        container = container_client
    
    filename = os.path.basename(local_path)
    blob_name = f"{blob_prefix}/{filename}" if blob_prefix else filename
    
    blob_client = container.get_blob_client(blob_name)
    file_size = os.path.getsize(local_path)
    
    with open(local_path, "rb") as f:
        blob_client.upload_blob(f, overwrite=True)
    
    print(f"  Uploaded: {blob_name} ({file_size / 1024:.1f} KB)")
    return blob_name

# Example: upload the sample PDF from the repo
sample_pdf = os.path.join("..", "sample_data", "ManitouCruise22_OG-0.pdf")
if os.path.exists(sample_pdf):
    uploaded_blob = upload_file(sample_pdf, blob_prefix="demo-upload")
    print(f"\nBlob path: {uploaded_blob}")
else:
    print(f"Sample file not found at {sample_pdf}. Update the path to your local file.")

### Batch Upload Multiple Files

Upload all files matching a glob pattern (e.g., all PDFs from a local folder).

In [ ]:
def batch_upload(local_folder, pattern="*.pdf", blob_prefix="", container=None):
    """Upload all files matching a pattern from a local folder.
    
    Args:
        local_folder: Path to the local folder containing files.
        pattern: Glob pattern for files to upload (default: *.pdf).
        blob_prefix: Optional folder prefix in the blob container.
        container: Container client (defaults to upload_container).
    
    Returns:
        List of uploaded blob names.
    """
    if container is None:
        container = container_client
    
    file_paths = glob.glob(os.path.join(local_folder, pattern))
    
    if not file_paths:
        print(f"No files matching '{pattern}' found in {local_folder}")
        return []
    
    print(f"Uploading {len(file_paths)} file(s) from {local_folder}...")
    uploaded = []
    for i, path in enumerate(file_paths, 1):
        blob_name = upload_file(path, blob_prefix=blob_prefix, container=container)
        uploaded.append(blob_name)
        # Progress indicator
        if i % 10 == 0 or i == len(file_paths):
            print(f"  Progress: {i}/{len(file_paths)}")
    
    print(f"\nBatch upload complete: {len(uploaded)} files uploaded.")
    return uploaded

# Example: upload all PDFs from the sample_data folder
uploaded_blobs = batch_upload(
    local_folder=os.path.join("..", "sample_data"),
    pattern="*.pdf",
    blob_prefix="demo-batch"
)
uploaded_blobs

---

## End-to-End: Upload, Ingest, and Verify

This section ties everything together:
1. Uploads a file to blob storage
2. Creates a test index
3. Triggers ingestion with test `entra_id` and `session_id` values
4. Polls until completion
5. Verifies indexed content

This demonstrates the full workflow a RAG chat application would use to onboard user-uploaded documents.

In [ ]:
import uuid

# ---- Test Identity Values ----
# In a real RAG chat app, these come from Entra ID authentication and the user's chat session.
# Here we use fake values for testing.
test_entra_id = f"test-user-{uuid.uuid4().hex[:8]}"
test_session_id = f"test-session-{uuid.uuid4().hex[:8]}"

print(f"Test Entra ID:  {test_entra_id}")
print(f"Test Session ID: {test_session_id}")

# ---- Settings for E2E test ----
e2e_index_stem = "e2e-test-index"
e2e_source_container = source_container  # from earlier cell
e2e_extract_container = extract_container
e2e_blob_prefix = f"e2e/{test_session_id}"

# ---- Step 1: Upload test file ----
print("\n--- Step 1: Upload test file ---")
sample_pdf = os.path.join("..", "sample_data", "ManitouCruise22_OG-0.pdf")
blob_name = upload_file(sample_pdf, blob_prefix=e2e_blob_prefix)

# ---- Step 2: Create test index ----
print("\n--- Step 2: Create test index ---")
create_index_uri = f"{function_uri}/api/create_new_index?code={function_key}"
create_index_payload = {
    "index_stem_name": e2e_index_stem,
    "fields": fields,
    "dimensions": embedding_dimensions
}
response = requests.post(create_index_uri, json=create_index_payload)
e2e_index_name = response.text
print(f"Created index: {e2e_index_name}")

# ---- Step 3: Trigger ingestion with test identity ----
print("\n--- Step 3: Trigger ingestion ---")
ingestion_uri = f"{function_uri}/api/orchestrators/pdf_orchestrator?code={function_key}"
ingestion_payload = {
    "source_container": e2e_source_container,
    "extract_container": e2e_extract_container,
    "prefix_path": blob_name,
    "index_name": e2e_index_name,
    "automatically_delete": True,
    "analyze_images": False,
    "chunking_strategy": "pagewise",
    "embedding_model": embedding_model,
    "cosmos_logging": True,
    "entra_id": test_entra_id,
    "session_id": test_session_id
}

response = requests.post(ingestion_uri, json=ingestion_payload)
e2e_status_uri = response.json()["statusQueryGetUri"]
print(f"Orchestrator started. Polling...")

# ---- Step 4: Poll until completion ----
print("\n--- Step 4: Polling orchestrator ---")
timeout_minutes = 15
deadline = time.time() + (timeout_minutes * 60)
final_status = "Unknown"

while time.time() < deadline:
    time.sleep(15)
    resp = requests.get(e2e_status_uri).json()
    runtime_status = resp.get("runtimeStatus", "Unknown")
    custom_status = resp.get("customStatus", "")
    
    if runtime_status in ("Completed", "Failed", "Terminated"):
        final_status = runtime_status
        clear_output(wait=True)
        break
    
    clear_output(wait=True)
    print(f"Status: {runtime_status} | {custom_status}")

if final_status == "Completed":
    print(f"Ingestion COMPLETED successfully!")
    print(f"  Entra ID:   {test_entra_id}")
    print(f"  Session ID: {test_session_id}")
    print(f"  Index:      {e2e_index_name}")
elif final_status == "Failed":
    print(f"Ingestion FAILED.")
    print(f"  Error: {resp.get('output', 'Unknown error')}")
else:
    print(f"Ingestion did not complete within {timeout_minutes} minutes. Last status: {final_status}")

# ---- Step 5: Verify indexed content ----
print("\n--- Step 5: Verify indexed content ---")
if final_status == "Completed":
    get_index_uri = f"{function_uri}/api/get_active_index?code={function_key}"
    resp = requests.post(get_index_uri, json={"index_stem_name": e2e_index_stem})
    print(f"Active index: {resp.text}")
    print("Documents were successfully indexed and are ready for RAG queries.")
else:
    print("Skipping verification (ingestion did not complete).")